# Badge 4 : DataLake WorkShop


**CATALOGUE DE VETEMENT DE SPORT**


## 📓 Cloud Folders for Staging and Storing Files

In [ ]:
%%sql -r dataframe_1
use role sysadmin;

create database ZENAS_ATHLEISURE_DB;
drop schema ZENAS_ATHLEISURE_DB.public;
create schema products;

## Lesson 3 📓 Working with non-load structured Data

In [ ]:
%%sql -r dataframe_2
list @zenas_athleisure_db.products.product_metadata;

In [ ]:
%%sql -r dataframe_10
use schema products;

In [ ]:
%%sql -r dataframe_3
select $1
from @product_metadata; 

In [ ]:
%%sql -r dataframe_4
select $1
from @product_metadata/product_coordination_suggestions.txt; 

In [ ]:
%%sql -r dataframe_5
select $1
from @product_metadata/sweatsuit_sizes.txt; 

In [ ]:
%%sql -r dataframe_6
use database zenas_athleisure_db;

create file format products.zmd_file_format_1
RECORD_DELIMITER = '^';


In [ ]:
%%sql -r dataframe_7
select $1
from @product_metadata/product_coordination_suggestions.txt
(file_format => zmd_file_format_1);

In [ ]:
%%sql -r dataframe_11
create file format zmd_file_format_2
FIELD_DELIMITER = '^';  


In [ ]:
%%sql -r dataframe_12
select $1,$2, $3, $4
from @product_metadata/product_coordination_suggestions.txt
(file_format => zmd_file_format_2);

In [ ]:
%%sql -r dataframe_13
create or replace file format zmd_file_format_3
FIELD_DELIMITER = '=' -- cree des colonnes
RECORD_DELIMITER = '^' -- cree des lignes 
TRIM_SPACE = TRUE -- supprime les espaces


In [ ]:
%%sql -r dataframe_14
select replace($1,chr(13)||chr(10)),replace($2,chr(13)||chr(10))
from @product_metadata/product_coordination_suggestions.txt
(file_format => zmd_file_format_3);

In [ ]:
%%sql -r sweatsuit_sizes
select $1
from @product_metadata/sweatsuit_sizes.txt; 

In [ ]:
%%sql -r dataframe_15
use database zenas_athleisure_db;

create or replace file format products.zmd_file_format_1
RECORD_DELIMITER = ';' -- colonnes
-- FIELD_DELIMITER= ' ' -- lignes
TRIM_SPACE = TRUE -- supprime les espaces


In [ ]:
%%sql -r dataframe_17
use schema products;

select replace($1,chr(13)||chr(10)) as sizes_available
from @product_metadata/sweatsuit_sizes.txt
(file_format => zmd_file_format_1)
where sizes_available <> '';

In [ ]:
%%sql -r dataframe_8
select $1
from @product_metadata/swt_product_line.txt

In [ ]:
%%sql -r dataframe_9
use database zenas_athleisure_db;

create or replace file format products.zmd_file_format_2
RECORD_DELIMITER = ';' -- enregistrement
FIELD_DELIMITER= '|' -- les colonnes de l'enregistrement
TRIM_SPACE = TRUE -- supprime les espaces

In [ ]:
%%sql -r dataframe_18
use schema products;

select replace($1,chr(13)||chr(10)) ,$2,$3 -- code ascii retour charriot et saut de ligne
from @product_metadata/swt_product_line.txt
(file_format => zmd_file_format_2);

### Creation view des produits


In [ ]:
%%sql -r dataframe_19
create view zenas_athleisure_db.products.sweatsuit_sizes as 
select replace($1,chr(13)||chr(10)) as sizes_available
from @product_metadata/sweatsuit_sizes.txt
(file_format => zmd_file_format_1)
where sizes_available <> '';

In [ ]:
%%sql -r dataframe_20
select * from zenas_athleisure_db.products.sweatsuit_sizes;

In [ ]:
%%sql -r dataframe_21
create or replace view zenas_athleisure_db.products.SWEATBAND_PRODUCT_LINE as 
select replace($1,chr(13)||chr(10)) as product_code, $2 as headband_description, $3 as  wristband_description -- code ascii retour charriot et saut de ligne
from @product_metadata/swt_product_line.txt
(file_format => zmd_file_format_2);

In [ ]:
%%sql -r dataframe_22
select * from zenas_athleisure_db.products.SWEATBAND_PRODUCT_LINE;

In [ ]:
%%sql -r dataframe_23
create or replace view zenas_athleisure_db.products.SWEATBAND_COORDINATION as
select replace($1,chr(13)||chr(10)) as product_code ,replace($2,chr(13)||chr(10)) as has_matching_sweatsuit
from @product_metadata/product_coordination_suggestions.txt
(file_format => zmd_file_format_3);

In [ ]:
%%sql -r dataframe_24
select * from zenas_athleisure_db.products.SWEATBAND_COORDINATION

## Lesson 4 📓 Working with non-load unstructured Data

### 📓 Unstructured Non-Loaded Data

In [ ]:
%%sql -r dataframe_26
list @sweatsuits; 

In [ ]:
%%sql -r dataframe_25
select $1
from @sweatsuits; 

In [ ]:
%%sql -r dataframe_27
select $1
from @sweatsuits/purple_sweatsuit.png; 

In [ ]:
%%sql -r dataframe_28
select metadata$filename, count(metadata$file_row_number) as number_of_rows
from @sweatsuits 
group by metadata$filename
order by 2 desc;



### 📓 File Formats? Nope, Directory Tables!

In [ ]:
%%sql -r dataframe_29
select * 
from directory(@sweatsuits); -- table de repertoire 


In [ ]:
%%sql -r dataframe_30
select INITCAP(REPLACE(REPLACE(relative_path, '_', ' '), '.png')) as product_name
from directory(@sweatsuits);

### 🥋 Can you Join Directory Tables with Other Tables?

In [ ]:
%%sql -r dataframe_31
--create an internal table for some sweatsuit info
create or replace table zenas_athleisure_db.products.sweatsuits (
	color_or_style varchar(25),
	file_name varchar(50),
	price number(5,2)
);

--fill the new table with some data
insert into  zenas_athleisure_db.products.sweatsuits 
          (color_or_style, file_name, price)
values
 ('Burgundy', 'burgundy_sweatsuit.png',65)
,('Charcoal Grey', 'charcoal_grey_sweatsuit.png',65)
,('Forest Green', 'forest_green_sweatsuit.png',64)
,('Navy Blue', 'navy_blue_sweatsuit.png',65)
,('Orange', 'orange_sweatsuit.png',65)
,('Pink', 'pink_sweatsuit.png',63)
,('Purple', 'purple_sweatsuit.png',64)
,('Red', 'red_sweatsuit.png',68)
,('Royal Blue',	'royal_blue_sweatsuit.png',65)
,('Yellow', 'yellow_sweatsuit.png',67);

In [ ]:
%%sql -r dataframe_32
select * from zenas_athleisure_db.products.sweatsuits as sw
inner join directory(@sweatsuits) as img
on sw.FILE_NAME = img.relative_path;

### 🥋 3 Way Join - CROSS JOIN Included!

In [ ]:
%%sql -r dataframe_33
create view zenas_athleisure_db.products.PRODUCT_LIST as
select INITCAP(REPLACE(REPLACE(img.relative_path, '_', ' '), '.png')) as product_name,
sw.file_name,
sw.color_or_style,
sw.price,
img.file_url
from zenas_athleisure_db.products.sweatsuits as sw
inner join directory(@sweatsuits) as img
on sw.FILE_NAME = img.relative_path;

In [ ]:
%%sql -r dataframe_34
select * 
from product_list p
cross join sweatsuit_sizes;

In [ ]:
%%sql -r dataframe_35
create view zenas_athleisure_db.products.catalog as 
select * 
from product_list p
cross join sweatsuit_sizes;

### 🥋 Do WHAT You Can, WHERE You Can, HOW You Can

In [ ]:
%%sql -r dataframe_36
-- Add a table to map the sweatsuits to the sweat band sets
create table zenas_athleisure_db.products.upsell_mapping
(
sweatsuit_color_or_style varchar(25)
,upsell_product_code varchar(10)
);

--populate the upsell table
insert into zenas_athleisure_db.products.upsell_mapping
(
sweatsuit_color_or_style
,upsell_product_code 
)
VALUES
('Charcoal Grey','SWT_GRY')
,('Forest Green','SWT_FGN')
,('Orange','SWT_ORG')
,('Pink', 'SWT_PNK')
,('Red','SWT_RED')
,('Yellow', 'SWT_YLW');

### 🥋 Kludges Galore? Or Relentless Resourcefulness?

In [ ]:
%%sql -r dataframe_37
-- Zena needs a single view she can query for her website prototype
create view catalog_for_website as 
select color_or_style
,price
,file_name
, get_presigned_url(@sweatsuits, file_name, 3600) as file_url
,size_list
,coalesce('Consider: ' ||  headband_description || ' & ' || wristband_description, 'Consider: White, Black or Grey Sweat Accessories')  as upsell_product_desc
from
(   select color_or_style, price, file_name
    ,listagg(sizes_available, ' | ') within group (order by sizes_available) as size_list
    from catalog
    group by color_or_style, price, file_name
) c
left join upsell_mapping u
on u.sweatsuit_color_or_style = c.color_or_style
left join sweatband_coordination sc
on sc.product_code = u.upsell_product_code
left join sweatband_product_line spl
on spl.product_code = sc.product_code;


In [ ]:
%%sql -r dataframe_38
select * from zenas_athleisure_db.products.catalog_for_website;

In [ ]:
%%sql -r dataframe_16
use database zenas_athleisure_db;
use schema PRODUCTS;
show streamlits;


describe streamlit "OY277C_BSNC95DZG"

In [ ]:
%%sql -r dataframe_39
show stages;

## Lesson 5 : MEL'S CONCEPT KICKOFF

In [ ]:
%%sql -r dataframe_40
use role sysadmin;
create database  MELS_SMOOTHIE_CHALLENGE_DB;
DROP SCHEMA  MELS_SMOOTHIE_CHALLENGE_DB.public;
CREATE SCHEMA  MELS_SMOOTHIE_CHALLENGE_DB.TRAILS;


In [ ]:
%%sql -r dataframe_41
create file format MELS_SMOOTHIE_CHALLENGE_DB.TRAILS.FF_JSON
    type = JSON
    --  [ formatTypeOptions ]
    -- comment = '<comment>'

In [ ]:
%%sql -r dataframe_42
create file format MELS_SMOOTHIE_CHALLENGE_DB.TRAILS.FF_PARQUET
    type = PARQUET

In [ ]:
%%sql -r dataframe_43
select *
from @trails_geojson
(file_format => ff_json);

In [ ]:
%%sql -r dataframe_44
select 
from @trails_parquet
(file_format => ff_parquet);

In [ ]:
%%sql -r dataframe_45
select 
$1:sequence_1::bigint as sequence_1,
$1:trail_name::varchar as trail_name,
$1:latitude::float as latitude,
$1:longitude::float as longitude,
$1:sequence_2::int as sequence_2,
$1:elevation as elevation
from @trails_parquet
(file_format => FF_PARQUET)
order by sequence_1;

In [ ]:
%%sql -r dataframe_46
--Nicely formatted trail data
select 
 $1:sequence_1 as point_id,
 $1:trail_name::varchar as trail_name,
 $1:latitude::number(11,8) as lng, --remember we did a gut check on this data
 $1:longitude::number(11,8) as lat
from @trails_parquet
(file_format => ff_parquet)
order by point_id;

In [ ]:
%%sql -r dataframe_47
create or replace view MELS_SMOOTHIE_CHALLENGE_DB.TRAILS.CHERRY_CREEK_TRAIL as 
select 
 $1:sequence_1 as point_id,
 $1:trail_name::varchar as trail_name,
 $1:latitude::number(11,8) as lng, --remember we did a gut check on this data
 $1:longitude::number(11,8) as lat
from @trails_parquet
(file_format => ff_parquet)
order by point_id;

### 🥋 Replace It With a Better View

In [ ]:
%%sql -r dataframe_48
--Using concatenate to prepare the data for plotting on a map
select top 100 
 lng||' '||lat as coord_pair
,'POINT('||coord_pair||')' as trail_point
from cherry_creek_trail;

In [ ]:
%%sql -r dataframe_49
--To add a column, we have to replace the entire view
--changes to the original are shown in red
create or replace view cherry_creek_trail as
select 
 $1:sequence_1 as point_id,
 $1:trail_name::varchar as trail_name,
 $1:latitude::number(11,8) as lng,
 $1:longitude::number(11,8) as lat,
 lng||' '||lat as coord_pair
from @trails_parquet
(file_format => ff_parquet)
order by point_id;

### 🥋 Can We Generate a LINESTRING( )?

In [ ]:
%%sql -r dataframe_50
select 
'LINESTRING('||
listagg(coord_pair, ',') 
within group (order by point_id)
||')' as my_linestring
from cherry_creek_trail
where point_id <= 10
group by trail_name;

### 🥋 Explore the geoJSON Files


In [ ]:
%%sql -r dataframe_51
select
$1:features[0]:properties:Name::string as feature_name
,$1:features[0]:geometry:coordinates::string as feature_coordinates
,$1:features[0]:geometry::string as geometry
,$1:features[0]:properties::string as feature_properties
,$1:crs:properties:name::string as specs
,$1 as whole_object
from @trails_geojson (file_format => ff_json);

### 🎯 Create a View for the geoJSON files


In [ ]:
%%sql -r dataframe_52
create or replace view MELS_SMOOTHIE_CHALLENGE_DB.TRAILS.DENVER_AREA_TRAILS as
select
$1:features[0]:properties:Name::string as feature_name
,$1:features[0]:geometry:coordinates::string as feature_coordinates
,$1:features[0]:geometry::string as geometry
,$1:features[0]:properties::string as feature_properties
,$1:crs:properties:name::string as specs
,$1 as whole_object
from @trails_geojson (file_format => ff_json);

## Lesson 7 : Exploring Geospatial Functions

In [ ]:
%%sql -r dataframe_53
use database MELS_SMOOTHIE_CHALLENGE_DB;
use schema trails;
select 
'LINESTRING('||
listagg(coord_pair, ',') 
within group (order by point_id)
||')' as my_linestring
,st_length(TO_GEOGRAPHY(my_linestring)) as length_of_trail
,round((length_of_trail/1000)) as length_of_trail_km
from cherry_creek_trail
group by trail_name;

In [ ]:
%%sql -r dataframe_54
use role sysadmin;
use database mels_smoothie_challenge_db; 
use schema trails;

select 
feature_name,
st_length(to_geography(geometry)) as geom_length,
whole_object:features[0]:geometry::string as wo_coordinates,
st_length(to_geography(wo_coordinates)) as wo_length
from denver_area_trails

In [ ]:
%%sql -r dataframe_55
create or replace view MELS_SMOOTHIE_CHALLENGE_DB.TRAILS.DENVER_AREA_TRAILS(
	FEATURE_NAME,
	FEATURE_COORDINATES,
	GEOMETRY,
    trail_length,
	FEATURE_PROPERTIES,
	SPECS,
	WHOLE_OBJECT
) as
select
$1:features[0]:properties:Name::string as feature_name
,$1:features[0]:geometry:coordinates::string as feature_coordinates
,$1:features[0]:geometry::string as geometry
,st_length(to_geography($1:features[0]:geometry::string)) as trail_length
,$1:features[0]:properties::string as feature_properties
,$1:crs:properties:name::string as specs
,$1 as whole_object
from @trails_geojson (file_format => ff_json);

In [ ]:
%%sql -r dataframe_57
select * from denver_area_trails;

In [ ]:
%%sql -r dataframe_56
--Create a view that will have similar columns to DENVER_AREA_TRAILS 
--Even though this data started out as Parquet, and we're joining it with geoJSON data
--So let's make it look like geoJSON instead.
create or replace view DENVER_AREA_TRAILS_2 as
select 
trail_name as feature_name
,'{"coordinates":['||listagg('['||lng||','||lat||']',',') within group (order by point_id)||'],"type":"LineString"}' as geometry
,st_length(to_geography(geometry))  as trail_length
from cherry_creek_trail
group by trail_name;

In [ ]:
%%sql -r dataframe_58
use database MELS_SMOOTHIE_CHALLENGE_DB;
use schema trails;
select * from denver_area_trails_2;

In [ ]:
%%sql -r dataframe_59
--Create a view that will have similar columns to DENVER_AREA_TRAILS 
select feature_name, geometry, to_geography(geometry) as goespatial_obj , trail_length
from DENVER_AREA_TRAILS
union all
select feature_name, geometry, to_geography(geometry) as goespatial_obj , trail_length
from DENVER_AREA_TRAILS_2;

In [ ]:
%%sql -r dataframe_60
--Add more GeoSpatial Calculations to get more GeoSpecial Information! 
select feature_name
, to_geography(geometry) as my_linestring
, st_xmin(my_linestring) as min_eastwest
, st_xmax(my_linestring) as max_eastwest
, st_ymin(my_linestring) as min_northsouth
, st_ymax(my_linestring) as max_northsouth
, trail_length
from DENVER_AREA_TRAILS
union all
select feature_name
, to_geography(geometry) as my_linestring
, st_xmin(my_linestring) as min_eastwest
, st_xmax(my_linestring) as max_eastwest
, st_ymin(my_linestring) as min_northsouth
, st_ymax(my_linestring) as max_northsouth
, trail_length
from DENVER_AREA_TRAILS_2;

In [ ]:
%%sql -r dataframe_61
use role sysadmin;
use database MELS_SMOOTHIE_CHALLENGE_DB;
use schema trails;

create or replace view trails_and_boundaries as
select feature_name
, to_geography(geometry) as my_linestring
, st_xmin(my_linestring) as min_eastwest
, st_xmax(my_linestring) as max_eastwest
, st_ymin(my_linestring) as min_northsouth
, st_ymax(my_linestring) as max_northsouth
, trail_length
from DENVER_AREA_TRAILS
union all
select feature_name
, to_geography(geometry) as my_linestring
, st_xmin(my_linestring) as min_eastwest
, st_xmax(my_linestring) as max_eastwest
, st_ymin(my_linestring) as min_northsouth
, st_ymax(my_linestring) as max_northsouth
, trail_length
from DENVER_AREA_TRAILS_2;

In [ ]:
%%sql -r dataframe_62
select 'POLYGON(('|| 
    min(min_eastwest)||' '||max(max_northsouth)||','|| 
    max(max_eastwest)||' '||max(max_northsouth)||','|| 
    max(max_eastwest)||' '||min(min_northsouth)||','|| 
    min(min_eastwest)||' '||min(min_northsouth)||'))' AS my_polygon
from trails_and_boundaries;

In [ ]:
%%sql -r dataframe_63
-- Melanie's Location into a 2 Variables (mc for melanies cafe)
set mc_lng='-104.97300245114094';
set mc_lat='39.76471253574085';

--Confluence Park into a Variable (loc for location)
set loc_lng='-105.00840763333615'; 
set loc_lat='39.754141917497826';

--Test your variables to see if they work with the Makepoint function
select st_makepoint($mc_lng,$mc_lat) as melanies_cafe_point;
select st_makepoint($loc_lng,$loc_lat) as confluent_park_point;

--use the variables to calculate the distance from 
--Melanie's Cafe to Confluent Park
select st_distance(
        st_makepoint($mc_lng,$mc_lat)
        ,st_makepoint($loc_lng,$loc_lat)
        ) as mc_to_cp;

In [ ]:
%%sql -r dataframe_64
use role sysadmin;
create schema mels_smoothie_challenge_db.locations;


In [ ]:
%%sql -r dataframe_65

CREATE OR REPLACE FUNCTION mels_smoothie_challenge_db.locations.distance_to_mc(loc_lng number(38,32),loc_lat number(38,32))
  RETURNS FLOAT
  AS
  $$
   st_distance(
        st_makepoint('-104.97300245114094','39.76471253574085')
        ,st_makepoint(loc_lng,loc_lat)
        )
  $$
  ;

In [ ]:
%%sql -r dataframe_66
--Tivoli Center into the variables 
set tc_lng='-105.00532059763648'; 
set tc_lat='39.74548137398218';

select distance_to_mc($tc_lng,$tc_lat);

In [ ]:
%%sql -r dataframe_67
select * 
from OPENSTREETMAP_DENVER.DENVER.V_OSM_DEN_AMENITY_SUSTENANCE
where 
    ((amenity in ('fast_food','cafe','restaurant','juice_bar'))
    and 
    (name ilike '%jamba%' or name ilike '%juice%'
     or name ilike '%superfruit%'))
 or 
    (cuisine like '%smoothie%' or cuisine like '%juice%');

In [ ]:
%%sql -r dataframe_68
use role sysadmin;
create or replace view mels_smoothie_challenge_db.locations.COMPETITION as 
select * 
from OPENSTREETMAP_DENVER.DENVER.V_OSM_DEN_AMENITY_SUSTENANCE
where 
    ((amenity in ('fast_food','cafe','restaurant','juice_bar'))
    and 
    (name ilike '%jamba%' or name ilike '%juice%'
     or name ilike '%superfruit%'))
 or 
    (cuisine like '%smoothie%' or cuisine like '%juice%');

In [ ]:
%%sql -r dataframe_69
SELECT
 name
 ,cuisine
 , ST_DISTANCE(
    st_makepoint('-104.97300245114094','39.76471253574085')
    , coordinates
  ) AS distance_to_melanies
 ,*
FROM  competition
ORDER by distance_to_melanies;

In [ ]:
%%sql -r dataframe_70
CREATE OR REPLACE FUNCTION distance_to_mc(lng_and_lat GEOGRAPHY)
  RETURNS FLOAT
  AS
  $$
   st_distance(
        st_makepoint('-104.97300245114094','39.76471253574085')
        ,lng_and_lat
        )
  $$
  ;

In [ ]:
%%sql -r dataframe_71
SELECT
 name
 ,cuisine
 ,distance_to_mc(coordinates) AS distance_to_melanies
 ,*
FROM  competition
ORDER by distance_to_melanies;

In [ ]:
%%sql -r dataframe_72
-- Tattered Cover Bookstore McGregor Square
set tcb_lng='-104.9956203'; 
set tcb_lat='39.754874';

--this will run the first version of the UDF
select distance_to_mc($tcb_lng,$tcb_lat);

--this will run the second version of the UDF, bc it converts the coords 
--to a geography object before passing them into the function
select distance_to_mc(st_makepoint($tcb_lng,$tcb_lat));

--this will run the second version bc the Sonra Coordinates column
-- contains geography objects already
select name
, distance_to_mc(coordinates) as distance_to_melanies 
, ST_ASWKT(coordinates)
from OPENSTREETMAP_DENVER.DENVER.V_OSM_DEN_SHOP
where shop='books' 
and name like '%Tattered Cover%'
and addr_street like '%Wazee%';

### Notion de fonction signature +> EN INFORMATIQUE POLYMORPHISME / SURCHARGE

### Analyze Potential Promotion Partners

In [ ]:
%%sql -r dataframe_73


In [ ]:
%%sql -r dataframe_74
select * from openstreetmap_denver.denver.V_OSM_DEN_SHOP_OUTDOORS_AND_SPORT_VEHICLES where shop = 'bicycle';

In [ ]:
%%sql -r dataframe_75
create view mels_smoothie_challenge_db.locations.denver_bike_shops as 
select 
name,
st_distance(st_makepoint('-104.97300245114094','39.76471253574085'), coordinates) as distance_to_melanies,
coordinates
from 
openstreetmap_denver.denver.V_OSM_DEN_SHOP_OUTDOORS_AND_SPORT_VEHICLES where shop = 'bicycle';

In [ ]:
%%sql -r dataframe_76
select * from mels_smoothie_challenge_db.locations.denver_bike_shops  
order by distance_to_melanies;

## Lesson 9 : Lions, and Tigers, and Bears, Oh My! 

**Notion de : 📓  Materialized Views, External Tables,  and Iceberg Tables**

Vue Materialisée bien si : gros calcul, frequemment intérrogée **MAISSSSS** ne change que rarement

In [ ]:
use role sysadmin;
use database mels_smoothie_challenge_db;
use schema trails;
create or replace external table T_CHERRY_CREEK_TRAIL(
my_filename varchar(100) as (metadata$filename::varchar(100))
)
location= @external_aws_dlkw
auto_refresh = true
file_format = (type = parquet);

In [ ]:
select * from t_cherry_creek_trail;

In [ ]:
create secure materialized view MELS_SMOOTHIE_CHALLENGE_DB.TRAILS.SMV_CHERRY_CREEK_TRAIL(
	POINT_ID,
	TRAIL_NAME,
	LNG,
	LAT,
	COORD_PAIR,
    DISTANCE_TO_MELANIES
) as
select 
 value:sequence_1 as point_id,
 value:trail_name::varchar as trail_name,
 value:latitude::number(11,8) as lng,
 value:longitude::number(11,8) as lat,
 lng||' '||lat as coord_pair,
 locations.distance_to_mc(st_makepoint(lng, lat)) as distance_to_melanies
from t_cherry_creek_trail;

## 🥋 Set Up Your Iceberg External Volume and User

In [ ]:
use role accountadmin;
CREATE OR REPLACE EXTERNAL VOLUME iceberg_external_volume
   STORAGE_LOCATIONS =
      (
         (
            NAME = 'iceberg-s3-us-west-2'
            STORAGE_PROVIDER = 'S3'
            STORAGE_BASE_URL = 's3://uni-dlkw-iceberg'
            STORAGE_AWS_ROLE_ARN = 'arn:aws:iam::321463406630:role/dlkw_iceberg_role'
            STORAGE_AWS_EXTERNAL_ID = 'dlkw_iceberg_id'
         )
      );

In [ ]:
DESC EXTERNAL VOLUME iceberg_external_volume;

## 🥋 Set Up Your Apache Iceberg DB and Table

In [ ]:
create database my_iceberg_db
  catalog = 'SNOWFLAKE'
  external_volume = 'iceberg_external_volume';
  

In [ ]:
set table_name = 'CCT_'||current_account();

create iceberg table identifier($table_name) (
    point_id number(10,0),
    trail_name string,
    coord_pair string,
    distance_to_melanies decimal(20,10),
    user_name string
)
  base_location = $table_name
  as
  select top 100
      point_id,
      trail_name,
      coord_pair,
      distance_to_melanies,
      current_user()
  from MELS_SMOOTHIE_CHALLENGE_DB.TRAILS.SMV_CHERRY_CREEK_TRAIL;

In [ ]:
set table_name = 'CCT_'||current_account();

create iceberg table identifier($table_name) (
    point_id number(10,0),
    trail_name string,
    coord_pair string,
    distance_to_melanies decimal(20,10),
    user_name string
)
  base_location = $table_name
  as
  select top 100
      point_id,
      trail_name,
      coord_pair,
      distance_to_melanies,
      current_user()
  from MELS_SMOOTHIE_CHALLENGE_DB.TRAILS.SMV_CHERRY_CREEK_TRAIL;

In [ ]:
select * from identifier($table_name);

In [ ]:
%%sql -r dataframe_88
update identifier($table_name)
set user_name = 'I am amazing!!'
where point_id = 1;